In [ ]:
"""
09_xgboost.py

Same setup as notebook 07 and 08, 
but with XGBoost on the financial ratios only.
Train/test split is by bookyear (2017-2018 train, 2019 test) - see
utils/time_split.py.
"""


'\n09_xgboost.py\n\nSame setup as 07/08, but with XGBoost on the financial ratios only.\nTrain/test split is by bookyear (2017-2018 train, 2019 test) - see\nutils/time_split.py.\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import FEATURES_FINANCIAL, RANDOM_STATE, TARGET, WINSOR_COLUMNS
from utils.model_results import save_model_results
from utils.print_section import print_section
from utils.time_split import time_based_split
from utils.winsorizer import Winsorizer

MODEL_NAME = "XGBoost Financial"

X_train, X_test, y_train, y_test = time_based_split(df, FEATURES_FINANCIAL, TARGET)

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()
scale_pos_weight = negative_class / positive_class

print_section("Class imbalance")
print(f"Negative class: {negative_class:,}")
print(f"Positive class: {positive_class:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

pipeline = Pipeline([
    ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    )),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print_section("Performance")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
}

for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

print_section("Confusion matrix")

cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print()
print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

print_section("Classification report")
print(classification_report(y_test, y_pred, zero_division=0))

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES_FINANCIAL,
    "importance": pipeline.named_steps["model"].feature_importances_,
}).sort_values(by="importance", ascending=False)

print(importances)

print_section("Highest predicted probabilities")

prob_df = pd.DataFrame({"actual": y_test.values, "probability": y_prob})
print(prob_df.sort_values(by="probability", ascending=False).head(20))

print_section("Model improvement over baseline")

baseline_rate = y_train.mean()
print(f"Baseline failure rate (train): {baseline_rate:.4%}")
print(f"Model precision: {metrics['precision']:.4%}")
print(f"Improvement factor: {metrics['precision'] / baseline_rate:.2f}x")

save_model_results(MODEL_NAME, metrics)



Class imbalance
Negative class: 654,506
Positive class: 2,203
scale_pos_weight: 297.10



Performance
accuracy  : 0.8211
precision : 0.0078
recall    : 0.7383
f1        : 0.0155
roc_auc   : 0.8631
pr_auc    : 0.0233

Confusion matrix
[[266481  58009]
 [   162    457]]

True Negatives : 266,481
False Positives: 58,009
False Negatives: 162
True Positives : 457

Classification report


              precision    recall  f1-score   support

           0       1.00      0.82      0.90    324490
           1       0.01      0.74      0.02       619

    accuracy                           0.82    325109
   macro avg       0.50      0.78      0.46    325109
weighted avg       1.00      0.82      0.90    325109


Feature importance
         feature  importance
2       solvency    0.282816
0  profitability    0.229518
3      structure    0.152383
1      liquidity    0.135277
5           size    0.114808
4        log_age    0.085197

Highest predicted probabilities
        actual  probability
24179        0     0.978726
163833       0     0.975329
111123       0     0.974451
238874       0     0.974372
239323       0     0.973161
135487       0     0.972774
150157       0     0.972656
150806       0     0.971974
108063       1     0.971505
55779        0     0.971211
244871       0     0.971131
220762       0     0.970461
141913       0     0.970254
246966       0     0.9698